# RNN Gold: обучение BUY/SELL в Google Colab

Ноутбук обучает две GRU-модели по CSV-файлам, проверяет качество и экспортирует ONNX-модели вместе с `RNN_Scaler.mqh`.

Порядок запуска: выполните ячейки сверху вниз. В ячейке загрузки можно выбрать Google Drive или загрузку файлов с компьютера.

In [1]:
import importlib.util
import subprocess
import sys

required_packages = {
    'onnx': 'onnx',
    'onnxscript': 'onnxscript',
    'tqdm': 'tqdm',
}
missing_packages = [
    package_name
    for import_name, package_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print(f'Устанавливаю отсутствующие библиотеки: {missing_packages}')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', *missing_packages
    ])
else:
    print('Все необходимые библиотеки уже установлены.')

Все необходимые библиотеки уже установлены.


In [2]:
import os
import glob
import io
import json
import sys
import random
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# Основные параметры
USE_GOOGLE_DRIVE = False
DRIVE_BASE = "/content/drive/MyDrive/RNN"
DATA_DIR = "/content/data"
OUTPUT_DIR = "/content/output"
ONNX_DIR = os.path.join(OUTPUT_DIR, "onnx")

N_FEATURES = 16
BATCH_SIZE = 64
SEQ_LEN = 3
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.25
LR = 0.002
CONF_THRESH = 0.45
TRAIN_EPOCHS = 100
NORM_WINDOW = 98
NORM_EPS = 1e-6

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ONNX_DIR, exist_ok=True)
print(f"Device: {DEVICE}")

Device: cuda


## Загрузка CSV

In [3]:
# Проверка CSV без зависимости от выполнения предыдущих ячеек.
import glob
import os

DATA_DIR = globals().get('DATA_DIR', '/content/data')

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"Папка DATA_DIR не найдена: {DATA_DIR}. "
        "Создайте папку data и поместите туда CSV-файлы."
    )

all_csvs = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))
buy_csvs = sorted(glob.glob(os.path.join(DATA_DIR, 'data_BUY*.csv')))
sell_csvs = sorted(glob.glob(os.path.join(DATA_DIR, 'data_SELL*.csv')))

print(f"DATA_DIR: {DATA_DIR}")
print(f"CSV-файлов в data: {len(all_csvs)}")
print("Все CSV:", [os.path.basename(path) for path in all_csvs])
print("BUY:", [os.path.basename(path) for path in buy_csvs])
print("SELL:", [os.path.basename(path) for path in sell_csvs])

if not buy_csvs or not sell_csvs:
    raise FileNotFoundError(
        "В папке data должны находиться файлы data_BUY*.csv и data_SELL*.csv"
    )

DATA_DIR: /content/data
CSV-файлов в data: 2
Все CSV: ['data_BUY_20200102_20251230.csv', 'data_SELL_20200102_20251230.csv']
BUY: ['data_BUY_20200102_20251230.csv']
SELL: ['data_SELL_20200102_20251230.csv']


## Подготовка данных

In [4]:
def adaptive_normalize(values, eps=1e-6):
    mean = values.mean(axis=1, keepdims=True)
    std = values.std(axis=1, ddof=0, keepdims=True)
    return (values - mean) / np.maximum(std, eps)


def load_and_split_csv(filepath, seq_len, norm_window, eps=1e-6):
    frame = pd.read_csv(filepath)
    if 'datetime' in frame.columns:
        parsed = pd.to_datetime(frame['datetime'], errors='coerce')
        if parsed.notna().all():
            frame = frame.assign(datetime=parsed).sort_values('datetime').reset_index(drop=True)

    feature_cols = [column for column in frame.columns if column.startswith('lag')]
    if 'label' not in frame.columns:
        raise KeyError(f'Column label not found in {filepath}')
    if not feature_cols or len(feature_cols) % N_FEATURES != 0:
        raise ValueError(f'Количество lag-колонок должно быть кратно {N_FEATURES}')

    n_bars = len(feature_cols) // N_FEATURES
    if seq_len > n_bars:
        raise ValueError(f'seq_len={seq_len} больше числа баров={n_bars}')
    if norm_window % n_bars != 0:
        raise ValueError(f'norm_window={norm_window} должен делиться на n_bars={n_bars}')

    rows_per_window = norm_window // n_bars
    if len(frame) <= rows_per_window:
        raise ValueError('Недостаточно строк для окна нормализации')
    raw = frame[feature_cols].to_numpy(dtype=np.float64).reshape(-1, n_bars, N_FEATURES)

    # sliding_window_view добавляет размер окна последней осью.
    # Переносим его в позицию времени до разворота и reshape.
    chunks = np.lib.stride_tricks.sliding_window_view(raw, rows_per_window, axis=0)
    chunks = np.moveaxis(chunks, -1, 1)
    chunks = chunks[:, ::-1]
    x_raw = chunks.reshape(chunks.shape[0], -1, N_FEATURES)
    x_raw = x_raw[1:]
    y = frame['label'].to_numpy(dtype=int)[rows_per_window:]

    labels = np.unique(y)
    if not np.array_equal(labels, np.arange(len(labels))):
        raise ValueError(f'Метки должны быть 0..N-1, получены: {labels.tolist()}')
    x = adaptive_normalize(x_raw, eps)[:, :seq_len, :].astype(np.float32)
    n_train = int(len(x) * 0.8)
    if n_train == 0 or n_train == len(x):
        raise ValueError('Нужно больше данных для train/test разделения')

    x_train, x_test = x[:n_train], x[n_train:]
    y_train, y_test = y[:n_train], y[n_train:]
    train_ds = TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train))
    test_ds = TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test))
    print(f'{os.path.basename(filepath)}: rows={len(frame)}, train={len(train_ds)}, test={len(test_ds)}, bars={n_bars}')
    print(f'classes={dict(zip(*np.unique(y, return_counts=True)))}')
    return (DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True),
            DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False),
            seq_len, N_FEATURES, len(labels))

## Модель и обучение

In [5]:
class GRUClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        hidden = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
        output, _ = self.gru(x, hidden)
        return self.fc(self.dropout(output[:, -1, :]))


def train_one_model(model, train_loader, epochs, lr):
    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr * 0.1)
    best_acc, best_state = 0.0, None
    for _ in tqdm(range(epochs), desc='Train'):
        model.train()
        correct = total = 0
        total_loss = 0.0
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(x_batch)
            correct += (logits.argmax(1) == y_batch).sum().item()
            total += len(x_batch)
        scheduler.step()
        accuracy = correct / total
        if accuracy > best_acc:
            best_acc = accuracy
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_acc, total_loss / total

## Метрики и экспорт

In [6]:
def evaluate_model(model, test_loader):
    model.eval()
    predictions, labels, probabilities = [], [], []
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            logits = model(x_batch.to(DEVICE))
            predictions.extend(logits.argmax(1).cpu().numpy())
            labels.extend(y_batch.numpy())
            probabilities.extend(torch.softmax(logits, 1)[:, 1].cpu().numpy())
    predictions, labels, probabilities = map(np.asarray, (predictions, labels, probabilities))
    tp = int(np.sum((predictions == 1) & (labels == 1)))
    tn = int(np.sum((predictions == 0) & (labels == 0)))
    fp = int(np.sum((predictions == 1) & (labels == 0)))
    fn = int(np.sum((predictions == 0) & (labels == 1)))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    accuracy = (tp + tn) / len(labels) if len(labels) else 0.0
    print(f'accuracy={accuracy:.4f}, precision={precision:.4f}, recall={recall:.4f}, f1={f1:.4f}')
    print(f'confusion matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}')
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1,
            'a1': recall, 'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn}


def export_to_onnx(model, seq_len, n_features, path):
    model = model.cpu().eval()
    temporary_path = path + '.tmp'
    torch.onnx.export(model, torch.randn(1, seq_len, n_features), temporary_path,
                      input_names=['input'], output_names=['output'],
                      dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
                      opset_version=18)
    import onnx
    onnx_model = onnx.load(temporary_path)
    onnx.save(onnx_model, path)
    if os.path.exists(temporary_path):
        os.remove(temporary_path)
    print(f'ONNX: {path} ({os.path.getsize(path) / 1024:.1f} KB)')

## Запуск обучения BUY и SELL

In [7]:
def train_one_model(model, train_loader):
    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRAIN_EPOCHS, eta_min=LR * 0.1)
    best_acc, best_state, last_loss = 0.0, None, 0.0
    for epoch in tqdm(range(TRAIN_EPOCHS), desc='Train'):
        model.train()
        total_loss = correct = total = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            logits = model(inputs)
            loss = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * inputs.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += inputs.size(0)
        last_loss = total_loss / total
        accuracy = correct / total
        scheduler.step()
        if accuracy > best_acc:
            best_acc = accuracy
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        tqdm.write(f'epoch {epoch + 1}: loss={last_loss:.4f}, acc={accuracy:.4f}')
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_acc, last_loss


def evaluate_model(model, test_loader):
    model.eval()
    predictions, labels = [], []
    with torch.no_grad():
        for inputs, batch_labels in test_loader:
            predictions.extend(model(inputs.to(DEVICE)).argmax(1).cpu().numpy())
            labels.extend(batch_labels.numpy())
    predictions, labels = np.asarray(predictions), np.asarray(labels)
    tp = int(((predictions == 1) & (labels == 1)).sum())
    tn = int(((predictions == 0) & (labels == 0)).sum())
    fp = int(((predictions == 1) & (labels == 0)).sum())
    fn = int(((predictions == 0) & (labels == 1)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    accuracy = (tp + tn) / len(labels) if len(labels) else 0.0
    metrics = {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1, 'a1': recall, 'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn}
    print(f'confusion matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}')
    print(f'accuracy={accuracy:.4f}, precision={precision:.4f}, recall={recall:.4f}, F1={f1:.4f}')
    return metrics


def export_to_onnx(model, seq_len, n_features, path):
    model.eval().to('cpu')
    temporary_path = path + '.tmp'
    with io.StringIO() as captured, _redirect_stdout(captured):
        torch.onnx.export(model, torch.randn(1, seq_len, n_features), temporary_path,
                          input_names=['input'], output_names=['output'],
                          dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
                          opset_version=18, dynamo=False)
    import onnx
    onnx_model = onnx.load(temporary_path)
    raw_bytes = onnx_model.SerializeToString()
    if b'.onnx.data' in raw_bytes:
        raise RuntimeError('ONNX export contains an external data reference')
    with open(path, 'wb') as output_file:
        output_file.write(raw_bytes)
    for candidate in (temporary_path, temporary_path + '.data', path + '.data'):
        if os.path.exists(candidate):
            os.remove(candidate)
    print(f'ONNX: {path} ({os.path.getsize(path) / 1024:.1f} KB)')


def _redirect_stdout(target):
    class Redirect:
        def __enter__(self):
            self.previous = sys.stdout
            sys.stdout = target
            return self
        def __exit__(self, *args):
            sys.stdout = self.previous
    return Redirect()


def generate_scaler_mqh(path):
    content = f'''//+
// RNN_Scaler.mqh - generated by rnn_train_colab.ipynb
//+
#define NN_BUY_NORM_WINDOW  {NORM_WINDOW}
#define NN_BUY_SEQ_LEN      {SEQ_LEN}
#define NN_SELL_NORM_WINDOW {NORM_WINDOW}
#define NN_SELL_SEQ_LEN     {SEQ_LEN}
#define NN_MAX_NORM_WINDOW  {NORM_WINDOW}
#define NN_FEATURES         {N_FEATURES}
#define NN_NORM_STD_EPS     {NORM_EPS}
#define NN_NORM_WINDOW NN_BUY_NORM_WINDOW
#define NN_SEQ_LEN     NN_BUY_SEQ_LEN
'''.replace('\n', '\n')
    with open(path, 'w', encoding='utf-8') as output_file:
        output_file.write(content)
    print(f'MQL5 scaler: {path}')

In [8]:
# Скачать результаты на компьютер
from google.colab import files
from zipfile import ZipFile

archive_path = '/content/rnn_output.zip'
with ZipFile(archive_path, 'w') as archive:
    for path in glob.glob(os.path.join(OUTPUT_DIR, '**', '*'), recursive=True):
        if os.path.isfile(path):
            archive.write(path, os.path.relpath(path, OUTPUT_DIR))
files.download(archive_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# RNN Gold: обучение BUY/SELL в Google Colab

Ноутбук повторяет пайплайн `rnn_train.py`: загрузка CSV, адаптивная нормализация, обучение двух GRU-моделей, оценка и экспорт ONNX/MQL5.

Запускайте ячейки по порядку. По умолчанию используется Google Drive: файлы должны находиться в `MyDrive/RNN/data`.

In [9]:
%pip -q install onnx onnxscript tqdm

import os
import glob
import io
import json
import sys
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch: 2.11.0+cu128
Device: cuda


## Конфигурация и данные

Для загрузки локальных CSV установите `USE_GOOGLE_DRIVE = False` и выполните ячейку: появятся два диалога загрузки BUY и SELL.

In [10]:
# Сначала используем CSV из локальной папки /content/data.
# Google Drive подключается только если локальных CSV нет.
USE_GOOGLE_DRIVE = False
DRIVE_BASE = '/content/drive/MyDrive/RNN'
DATA_DIR = '/content/data'

local_csvs = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))
if not local_csvs:
    print(f'В {DATA_DIR} CSV не найдены. Подключаю Google Drive...')
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = DRIVE_BASE
    DATA_DIR = os.path.join(BASE, 'data')
else:
    BASE = '/content'
    print(f'Использую локальные CSV из {DATA_DIR}')

OUTPUT_DIR = os.path.join(BASE, 'output')
ONNX_DIR = os.path.join(OUTPUT_DIR, 'onnx')
os.makedirs(ONNX_DIR, exist_ok=True)

N_FEATURES = 16
BATCH_SIZE = 64
SEQ_LEN = 3
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.25
LR = 0.002
TRAIN_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 12
MIN_DELTA = 1e-4
NORM_WINDOW = 98
NORM_EPS = 1e-6
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

buy_csvs = sorted(glob.glob(os.path.join(DATA_DIR, 'data_BUY*.csv')))
sell_csvs = sorted(glob.glob(os.path.join(DATA_DIR, 'data_SELL*.csv')))
print(f'BASE: {BASE}')
print(f'DATA_DIR: {DATA_DIR}')
print(f'TRAIN_EPOCHS: {TRAIN_EPOCHS}')
print(f'EARLY_STOPPING_PATIENCE: {EARLY_STOPPING_PATIENCE}')
print('BUY:', [os.path.basename(path) for path in buy_csvs])
print('SELL:', [os.path.basename(path) for path in sell_csvs])
if not buy_csvs or not sell_csvs:
    raise FileNotFoundError(
        'В папке data должны находиться data_BUY*.csv и data_SELL*.csv'
    )

Использую локальные CSV из /content/data
BASE: /content
DATA_DIR: /content/data
TRAIN_EPOCHS: 100
EARLY_STOPPING_PATIENCE: 12
BUY: ['data_BUY_20200102_20251230.csv']
SELL: ['data_SELL_20200102_20251230.csv']


## Функции подготовки данных и модель

In [11]:
def adaptive_normalize(values, eps=1e-6):
    mean = values.mean(axis=1, keepdims=True)
    std = values.std(axis=1, ddof=0, keepdims=True)
    std = np.maximum(std, eps)
    return (values - mean) / std


def load_and_split_csv(filepath, seq_len, norm_window, eps=1e-6):
    dataframe = pd.read_csv(filepath)
    if 'datetime' in dataframe.columns:
        parsed = pd.to_datetime(dataframe['datetime'], errors='coerce')
        if parsed.notna().all():
            dataframe = dataframe.assign(datetime=parsed).sort_values('datetime').reset_index(drop=True)

    feature_cols = [column for column in dataframe.columns if column.startswith('lag')]
    if 'label' not in dataframe.columns:
        raise KeyError(f'Column label not found in {filepath}')
    if len(feature_cols) == 0 or len(feature_cols) % N_FEATURES != 0:
        raise ValueError(f'Количество lag-колонок должно быть кратно {N_FEATURES}')

    n_bars = len(feature_cols) // N_FEATURES
    if seq_len > n_bars:
        raise ValueError(f'seq_len={seq_len} больше числа баров={n_bars}')
    if norm_window % n_bars != 0:
        raise ValueError(f'norm_window={norm_window} должен делиться на n_bars={n_bars}')

    rows_per_window = norm_window // n_bars
    if len(dataframe) <= rows_per_window:
        raise ValueError('Недостаточно строк для окна нормализации')
    raw = dataframe[feature_cols].to_numpy(dtype=np.float64).reshape(-1, n_bars, N_FEATURES)

    windows = np.lib.stride_tricks.sliding_window_view(raw, rows_per_window, axis=0)
    windows = np.moveaxis(windows, -1, 1)
    windows = windows[:, ::-1]
    raw_sequences = windows.reshape(windows.shape[0], -1, N_FEATURES)
    raw_sequences = raw_sequences[1:]

    labels = dataframe['label'].to_numpy(dtype=int)[rows_per_window:]
    features = adaptive_normalize(raw_sequences, eps)[:, :seq_len, :].astype(np.float32)
    if not np.isin(labels, [0, 1]).all():
        raise ValueError('label must contain only 0 and 1')

    train_end = int(len(features) * 0.64)
    val_end = int(len(features) * 0.80)
    if train_end == 0 or val_end <= train_end or val_end == len(features):
        raise ValueError('Нужно больше данных для train/validation/test разделения')

    x_train, x_val, x_test = features[:train_end], features[train_end:val_end], features[val_end:]
    y_train, y_val, y_test = labels[:train_end], labels[train_end:val_end], labels[val_end:]
    train_ds = TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train))
    val_ds = TensorDataset(torch.from_numpy(x_val), torch.from_numpy(y_val))
    test_ds = TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test))
    print(f'{os.path.basename(filepath)}: rows={len(dataframe)}, train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}, bars={n_bars}')
    print(f'classes={dict(zip(*np.unique(labels, return_counts=True)))}')
    return (DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True),
            DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False),
            DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False), seq_len, N_FEATURES, len(np.unique(labels)))


class GRUClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes=2, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, inputs):
        hidden = torch.zeros(self.num_layers, inputs.size(0), self.hidden_size, device=inputs.device)
        outputs, _ = self.gru(inputs, hidden)
        return self.fc(self.dropout(outputs[:, -1, :]))

## Обучение, оценка и экспорт

In [12]:
def train_one_model(model, train_loader, val_loader):
    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=TRAIN_EPOCHS, eta_min=LR * 0.1
    )
    best_val_loss = float('inf')
    best_state = None
    best_epoch = 0
    patience_counter = 0

    for epoch in tqdm(range(TRAIN_EPOCHS), desc='Train'):
        model.train()
        train_loss_sum = 0.0
        train_correct = 0
        train_samples = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(inputs)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item() * len(inputs)
            train_correct += (logits.argmax(1) == labels).sum().item()
            train_samples += len(inputs)

        model.eval()
        val_loss_sum = 0.0
        val_correct = 0
        val_samples = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                logits = model(inputs)
                val_loss_sum += criterion(logits, labels).item() * len(inputs)
                val_correct += (logits.argmax(1) == labels).sum().item()
                val_samples += len(inputs)

        train_loss = train_loss_sum / train_samples
        train_accuracy = train_correct / train_samples
        val_loss = val_loss_sum / val_samples
        val_accuracy = val_correct / val_samples
        scheduler.step()

        tqdm.write(
            f'epoch {epoch + 1}: train_loss={train_loss:.4f}, '
            f'train_acc={train_accuracy:.4f}, val_loss={val_loss:.4f}, '
            f'val_acc={val_accuracy:.4f}'
        )

        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(
                f'Early stopping: epoch {epoch + 1}; '
                f'best epoch {best_epoch}; best val_loss={best_val_loss:.4f}'
            )
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_epoch, best_val_loss

## Запуск обучения

In [13]:
def train_and_export(csv_path, mode_name):
    loaded_data = load_and_split_csv(csv_path, SEQ_LEN, NORM_WINDOW, NORM_EPS)
    train_loader, val_loader, test_loader, seq_len, n_features = loaded_data[:5]
    num_classes = loaded_data[5] if len(loaded_data) > 5 else 2
    model = GRUClassifier(n_features, HIDDEN_SIZE, NUM_LAYERS, num_classes, DROPOUT)
    print(f'{mode_name}: parameters={sum(parameter.numel() for parameter in model.parameters()):,}')
    model, best_epoch, best_val_loss = train_one_model(model, train_loader, val_loader)
    metrics = evaluate_model(model, test_loader)
    onnx_path = os.path.join(ONNX_DIR, f'{mode_name.lower()}_model.onnx')
    export_to_onnx(model, seq_len, n_features, onnx_path)
    return {'csv': csv_path, 'seq_len': seq_len, 'n_features': n_features,
            'best_epoch': best_epoch, 'best_val_loss': best_val_loss,
            'test_metrics': metrics, 'onnx_path': onnx_path}

buy_csvs = sorted(glob.glob(os.path.join(DATA_DIR, 'data_BUY*.csv')))
sell_csvs = sorted(glob.glob(os.path.join(DATA_DIR, 'data_SELL*.csv')))
if not buy_csvs or not sell_csvs:
    raise FileNotFoundError('Нужны оба файла: data_BUY*.csv и data_SELL*.csv')

buy_result = train_and_export(buy_csvs[0], 'BUY')
sell_result = train_and_export(sell_csvs[0], 'SELL')
scaler_path = os.path.join(OUTPUT_DIR, 'RNN_Scaler.mqh')
generate_scaler_mqh(scaler_path)

summary = {'params': {'seq_len': SEQ_LEN, 'hidden_size': HIDDEN_SIZE, 'num_layers': NUM_LAYERS,
                      'dropout': DROPOUT, 'lr': LR, 'train_epochs': TRAIN_EPOCHS,
                      'early_stopping_patience': EARLY_STOPPING_PATIENCE,
                      'min_delta': MIN_DELTA, 'batch_size': BATCH_SIZE,
                      'norm_window': NORM_WINDOW, 'n_features': N_FEATURES},
           'buy_model': buy_result, 'sell_model': sell_result}
summary_path = os.path.join(OUTPUT_DIR, 'summary.json')
with open(summary_path, 'w', encoding='utf-8') as output_file:
    json.dump(summary, output_file, indent=2, ensure_ascii=False, default=float)

print('\n' + '=' * 60)
print('ФИНАЛЬНЫЕ МЕТРИКИ')
print('=' * 60)
for mode_name, result in [('BUY', buy_result), ('SELL', sell_result)]:
    metrics = result['test_metrics']
    print(f'\n{mode_name}: best_epoch={result["best_epoch"]}, val_loss={result["best_val_loss"]:.4f}')
    print(f'Accuracy={metrics["accuracy"]:.4f}, Precision={metrics["precision"]:.4f}, Recall={metrics["recall"]:.4f}, F1={metrics["f1"]:.4f}')
    print(f'TN={metrics["tn"]}, FP={metrics["fp"]}, FN={metrics["fn"]}, TP={metrics["tp"]}')
print(f'\nSummary: {summary_path}')
print('Готово. Файлы находятся в:', OUTPUT_DIR)

data_BUY_20200102_20251230.csv: rows=15974, train=10214, val=2554, test=3192, bars=7
classes={np.int64(0): np.int64(8393), np.int64(1): np.int64(7567)}
BUY: parameters=40,834


Train:   0%|          | 0/100 [00:00<?, ?it/s]

epoch 1: train_loss=0.6923, train_acc=0.5254, val_loss=0.6952, val_acc=0.4898
epoch 2: train_loss=0.6902, train_acc=0.5323, val_loss=0.6967, val_acc=0.4918
epoch 3: train_loss=0.6899, train_acc=0.5361, val_loss=0.6954, val_acc=0.5070
epoch 4: train_loss=0.6899, train_acc=0.5364, val_loss=0.6942, val_acc=0.5008
epoch 5: train_loss=0.6900, train_acc=0.5393, val_loss=0.6962, val_acc=0.4914
epoch 6: train_loss=0.6896, train_acc=0.5389, val_loss=0.7038, val_acc=0.4914
epoch 7: train_loss=0.6894, train_acc=0.5388, val_loss=0.6995, val_acc=0.4871
epoch 8: train_loss=0.6895, train_acc=0.5383, val_loss=0.6974, val_acc=0.4969
epoch 9: train_loss=0.6892, train_acc=0.5413, val_loss=0.6954, val_acc=0.5000
epoch 10: train_loss=0.6888, train_acc=0.5436, val_loss=0.6943, val_acc=0.4973
epoch 11: train_loss=0.6887, train_acc=0.5447, val_loss=0.6945, val_acc=0.5086
epoch 12: train_loss=0.6886, train_acc=0.5427, val_loss=0.6972, val_acc=0.4961
epoch 13: train_loss=0.6879, train_acc=0.5430, val_loss=0.696

Train:   0%|          | 0/100 [00:00<?, ?it/s]

epoch 1: train_loss=0.6922, train_acc=0.5300, val_loss=0.7000, val_acc=0.5261
epoch 2: train_loss=0.6914, train_acc=0.5307, val_loss=0.6910, val_acc=0.5294
epoch 3: train_loss=0.6906, train_acc=0.5342, val_loss=0.6909, val_acc=0.5352
epoch 4: train_loss=0.6903, train_acc=0.5324, val_loss=0.6927, val_acc=0.5195
epoch 5: train_loss=0.6897, train_acc=0.5367, val_loss=0.6917, val_acc=0.5327
epoch 6: train_loss=0.6901, train_acc=0.5340, val_loss=0.6942, val_acc=0.5140
epoch 7: train_loss=0.6896, train_acc=0.5381, val_loss=0.6912, val_acc=0.5294
epoch 8: train_loss=0.6894, train_acc=0.5399, val_loss=0.6935, val_acc=0.5308
epoch 9: train_loss=0.6898, train_acc=0.5387, val_loss=0.6917, val_acc=0.5275
epoch 10: train_loss=0.6890, train_acc=0.5386, val_loss=0.6923, val_acc=0.5305
epoch 11: train_loss=0.6881, train_acc=0.5459, val_loss=0.6935, val_acc=0.5176
epoch 12: train_loss=0.6881, train_acc=0.5442, val_loss=0.6926, val_acc=0.5338
epoch 13: train_loss=0.6870, train_acc=0.5430, val_loss=0.694

In [14]:
from google.colab import files

print('Файлы для установки в MT5:')
for path in (buy_result['onnx_path'], sell_result['onnx_path'], scaler_path, summary_path):
    print(path)

# Для скачивания по одному файлу раскомментируйте нужную строку:
# files.download(buy_result['onnx_path'])
# files.download(sell_result['onnx_path'])
# files.download(scaler_path)
# files.download(summary_path)

Файлы для установки в MT5:
/content/output/onnx/buy_model.onnx
/content/output/onnx/sell_model.onnx
/content/output/RNN_Scaler.mqh
/content/output/summary.json
